# ⚡ Análise de Consumo de Energia Elétrica no Brasil

## 1. Introdução
Este notebook apresenta a análise exploratória da base simulada de consumo de energia elétrica no Brasil, seguindo o Tema 14. O estudo busca compreender evolução temporal, diferenças regionais e setoriais, sazonalidade, demanda, temperatura, eficiência energética e emissões de CO₂.

## 2. Contextualização energética
O consumo de energia é um indicador associado à atividade econômica, urbanização, comportamento de consumo e eficiência energética. O projeto foi definido para investigar dados entre 2015 e 2024.

## 3. Explicação da base
A base contém ano, mês, data, região, UF, setor de consumo, consumo em MWh, demanda de pico, temperatura média, tarifa média, população, eficiência energética, emissão de CO₂ e nível de demanda.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../dados/simulacao_consumo_energia_brasil.csv')
df['data'] = pd.to_datetime(df['data'])
df.head()

In [ ]:
df.info()
df.describe(include='all').T

## 4. Leitura, limpeza e preparação

In [ ]:
df.isna().sum()

# Tipos e campos derivados
df['ano'] = df['data'].dt.year
df['mes'] = df['data'].dt.month
df['consumo_per_capita'] = df['consumo_mwh'] / df['populacao']
df['intensidade_co2'] = df['emissao_co2'] / df['consumo_mwh']
df.duplicated().sum()

## 5. Engenharia de atributos
Foram criados `consumo_per_capita` e `intensidade_co2`, além de garantir ano e mês derivados da data para análises temporais.

## 6. Análise exploratória

In [ ]:
print('Consumo total:', df['consumo_mwh'].sum())
print('UF com maior consumo:', df.groupby('uf')['consumo_mwh'].sum().idxmax())
print('Setor com maior consumo:', df.groupby('setor_consumo')['consumo_mwh'].sum().idxmax())
print('Demanda média:', df['demanda_pico'].mean())
print('Tarifa média:', df['tarifa_media'].mean())
print('Emissão total de CO2:', df['emissao_co2'].sum())

## 7. KPIs

In [ ]:
kpis = {
 'consumo_total_mwh': df.consumo_mwh.sum(),
 'uf_lider': df.groupby('uf').consumo_mwh.sum().idxmax(),
 'setor_lider': df.groupby('setor_consumo').consumo_mwh.sum().idxmax(),
 'demanda_media': df.demanda_pico.mean(),
 'tarifa_media': df.tarifa_media.mean(),
 'emissao_total_co2': df.emissao_co2.sum(),
 'eficiencia_media': df.eficiencia_energetica.mean()
}
kpis

## 8. Visualizações

In [ ]:
mensal=df.groupby('data',as_index=False).consumo_mwh.sum()
sns.lineplot(data=mensal,x='data',y='consumo_mwh')
plt.title('Evolução do consumo')
plt.xticks(rotation=45); plt.show()

In [ ]:
estado=df.groupby('uf')['consumo_mwh'].sum().sort_values(ascending=False).head(15)
sns.barplot(x=estado.values,y=estado.index)
plt.title('Ranking de estados'); plt.show()

In [ ]:
setor=df.groupby('setor_consumo')['consumo_mwh'].sum().sort_values(ascending=False)
sns.barplot(x=setor.index,y=setor.values)
plt.xticks(rotation=20); plt.title('Consumo por setor'); plt.show()

In [ ]:
sns.scatterplot(data=df.sample(min(len(df),1500),random_state=42),x='temperatura_media',y='consumo_mwh',hue='setor_consumo')
plt.title('Temperatura x consumo'); plt.show()

In [ ]:
heat=df.pivot_table(index='ano',columns='mes',values='consumo_mwh',aggfunc='sum')
sns.heatmap(heat,cmap='YlOrRd')
plt.title('Heatmap de sazonalidade'); plt.show()

In [ ]:
corr=df[['consumo_mwh','demanda_pico','temperatura_media','tarifa_media','populacao','eficiencia_energetica','emissao_co2']].corr()
sns.heatmap(corr,annot=True,cmap='RdBu_r',center=0)
plt.title('Matriz de correlação'); plt.show()

## 9. Interpretação dos resultados
Os resultados devem ser interpretados considerando o recorte temporal e os filtros analíticos. O ranking estadual identifica os maiores volumes de consumo; a análise setorial mostra os setores predominantes; a série temporal permite observar crescimento ou redução; e as análises de temperatura, demanda, eficiência e emissões ajudam a contextualizar o comportamento energético.

## 10. Conclusão
A base permite construir uma visão integrada do consumo de energia elétrica, combinando indicadores de volume, demanda, custo, eficiência e impacto ambiental. O dashboard complementa o notebook ao permitir filtros e exploração dinâmica dos resultados.